# E2B Example: Parallel SARIMA Grid Search

This notebook fans a bounded SARIMA search space out across multiple isolated E2B sandboxes.

Each worker sandbox gets the same monthly time series and one disjoint batch of candidate configurations. Inside the sandbox, the worker installs `statsmodels` if needed, fits its batch, writes artifacts, and returns structured metrics back to the notebook kernel.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "src" / "agents").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "agents").exists():
            repo_root = candidate
            break
        if (candidate / "openai-agents-python-preview" / "src" / "agents").exists():
            repo_root = candidate / "openai-agents-python-preview"
            break

src_root = repo_root / "src"
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

env_path = None
for candidate in [Path.cwd(), *Path.cwd().parents, repo_root, *repo_root.parents]:
    maybe_env = candidate / ".env"
    if maybe_env.exists():
        env_path = maybe_env
        break

if env_path is not None:
    try:
        from dotenv import load_dotenv

        load_dotenv(env_path, override=False)
    except Exception:
        for raw_line in env_path.read_text().splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

required_modules = ["agents", "openai", "e2b", "e2b_code_interpreter"]
missing_modules = [
    module_name for module_name in required_modules if importlib.util.find_spec(module_name) is None
]
if missing_modules:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root) + "[e2b]"]
    )

In [ ]:
from agents.extensions.sandbox import E2BSandboxType
from examples.sandbox.extensions.e2b.sarima_grid_search_parallel import (
    DEFAULT_CANDIDATE_BATCHES,
    build_dataset_csv,
    ranked_rows,
    require_credentials,
    run_parallel_grid_search,
)

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4")
SANDBOX_TYPE = E2BSandboxType(os.getenv("E2B_SANDBOX_TYPE", E2BSandboxType.E2B.value))
TEMPLATE = os.getenv("E2B_TEMPLATE") or None
TIMEOUT_SECONDS = int(os.getenv("E2B_SARIMA_TIMEOUT", "900"))
MAX_CONCURRENCY = int(os.getenv("E2B_SARIMA_MAX_CONCURRENCY", "3"))

print(build_dataset_csv().splitlines()[:8])
print(f"candidate batches: {len(DEFAULT_CANDIDATE_BATCHES)}")

In [ ]:
require_credentials()

batch_results = await run_parallel_grid_search(  # type: ignore[top-level-await]  # noqa: F704
    model=MODEL,
    sandbox_type=SANDBOX_TYPE,
    template=TEMPLATE,
    timeout_seconds=TIMEOUT_SECONDS,
    max_concurrency=MAX_CONCURRENCY,
)
batch_results

In [ ]:
rows = ranked_rows(batch_results)
try:
    import pandas as pd

    display(pd.DataFrame(rows))
except Exception:
    print(rows)

In [ ]:
champion = rows[0]
champion

## Notes

- This example uses one sandbox per candidate batch, not one sandbox per candidate. That keeps startup overhead reasonable while still showing true concurrent search.
- For a faster production example, preinstall `numpy`, `pandas`, `matplotlib`, and `statsmodels` into an E2B template and pass that template alias through `E2B_TEMPLATE`.
- The reduction step stays in the notebook kernel, which is where ranking, charting, and downstream model selection usually belong.
